# ToonCrafter on Colab (free T4)

Cartoon interpolation between two keyframes. Upload 2 images
(start/end frame of the shot) and get a 16-frame animated clip.
Chain clips for longer shots; upscale output with Real-ESRGAN if
needed (512x320 source).

**Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 https://github.com/Doubiiu/ToonCrafter /content/ToonCrafter
%cd /content/ToonCrafter
!pip -q install -r requirements.txt decord omegaconf einops pytorch_lightning kornia imageio

In [ ]:
# Download ToonCrafter_512 fp16 weights (fits T4 16GB)
import os
os.makedirs('checkpoints/tooncrafter_512_interp_v1', exist_ok=True)
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
p = hf_hub_download('Doubiiu/ToonCrafter', 'model.ckpt',
                    subfolder='tooncrafter_512_interp_v1',
                    local_dir='checkpoints/tooncrafter_512_interp_v1')
print('weights:', p)

In [ ]:
# Upload start + end frame images
from google.colab import files
os.makedirs('/content/in', exist_ok=True)
up = files.upload()  # pick frame_a.png and frame_b.png
for n in up:
    os.rename(n, '/content/in/' + n)
print(sorted(os.listdir('/content/in')))

In [ ]:
# Run interpolation -> /content/out/clip.mp4
# ToonCrafter's eval config: prompts + image pair via gradio or
scripts/eval/inference_i2v.py — adjust flags to match your clone.
!python scripts/eval/inference_i2v.py \
  --prompt 'an animated scene, smooth cartoon motion' \
  --image1 /content/in/frame_a.png \
  --image2 /content/in/frame_b.png \
  --steps 50 --cfg 7.5 --eta 1.0 --save_dir /content/out || true
# If the flag layout differs in this revision, fall back to the
# bundled gradio demo:
#   !python gradio_app.py --share
print(os.listdir('/content/out') if os.path.isdir('/content/out') else 'check script flags')

In [ ]:
# Download the generated clip(s)
from google.colab import files
for f in sorted(os.listdir('/content/out')):
    files.download('/content/out/' + f)